# Chapter 8 — Text Data Analysis: Security Logs & Threat Intelligence

## 📋 Latihan Soal

Analisis data teks dari berbagai sumber security menggunakan NLP techniques.

### Dataset
- 200 Firewall Log Messages (BLOCKED/ALLOWED/ALERT)
- 80 CVE Descriptions (RCE, SQLi, XSS, Buffer Overflow, dll)
- 60 TTP Descriptions (APT, Ransomware, Cyber Espionage, dll)
- 150 SIEM Alert Descriptions (Brute Force, Exfiltration, Malware, dll)

### Teknik
Text cleaning, stop words, POS tagging, stemming, lemmatisation, ngrams, word clouds, TF-IDF, sentiment analysis, LDA topic modelling, coherence score

In [ ]:
# ============================================
# SETUP: Download data NLTK & generate text dummy
# ============================================
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

import pandas as pd
import numpy as np
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from collections import Counter

# === Generate Text Dummy Security ===
np.random.seed(42)

# 1. Firewall Log Messages
firewall_templates = [
    "BLOCKED inbound TCP connection from {src} to {dst}:{port} protocol HTTPS detected on interface eth0 with payload anomaly",
    "ALLOWED outbound UDP traffic from {src} to external DNS server {dns} port 53 query resolved successfully",
    "BLOCKED suspicious ICMP echo request from {src} targeting internal subnet 192.168.1.0/24 possible reconnaissance activity",
    "ALERT repeated SYN flood detected from {src} to {dst}:{port} connection rate exceeded threshold possible DoS attack",
    "BLOCKED unauthorized RDP connection attempt from {src} to {dst}:3389 brute force pattern detected from external source",
    "ALLOWED encrypted TLS 1.3 session established from {src} to {dst}:{port} certificate validated successfully",
    "ALERT port scan detected from {src} targeting multiple hosts on subnet sequential connection attempts on ports 21 22 23 80 443",
    "BLOCKED malicious payload detected in HTTP POST request from {src} to {dst}:{port} signature match known exploit pattern",
    "BLOCKED DNS query to known C2 domain blocked from {src} threat intelligence match indicator of compromise detected",
    "ALERT lateral movement detected internal host {src} connecting to multiple hosts on SMB port 445 possible worm propagation",
]

src_ips = [f'203.45.{np.random.randint(0,256)}.{np.random.randint(1,255)}' for _ in range(50)]
dst_ips = [f'192.168.1.{np.random.randint(1,255)}' for _ in range(30)]
dns_servers = ['8.8.8.8', '1.1.1.1', '8.8.4.4', '9.9.9.9']
ports = [80, 443, 22, 3389, 8080, 3306, 445, 25, 53, 21]

firewall_logs = []
for _ in range(200):
    template = np.random.choice(firewall_templates)
    log = template.format(
        src=np.random.choice(src_ips),
        dst=np.random.choice(dst_ips),
        port=np.random.choice(ports),
        dns=np.random.choice(dns_servers)
    )
    firewall_logs.append(log)

# 2. CVE Descriptions
cve_templates = [
        "Remote code execution vulnerability in {product} version {ver} allows authenticated attacker to execute arbitrary commands via crafted input in {component} module resulting in complete system compromise",
        "SQL injection vulnerability in {product} web application allows remote attacker to extract sensitive database contents through unsanitized user input in search parameter",
        "Cross-site scripting vulnerability in {product} administration interface enables attacker to inject malicious JavaScript through stored comment field affecting all authenticated users",
        "Authentication bypass vulnerability in {product} API endpoint allows unauthenticated remote attacker to access restricted resources by manipulating session token validation logic",
        "Buffer overflow vulnerability in {product} network service parser allows remote attacker to execute arbitrary code via malformed packet triggering heap corruption in {component}",
        "Privilege escalation vulnerability in {product} local service allows authenticated low-privileged user to gain administrative access through improper access control validation",
        "Information disclosure vulnerability in {product} error handling mechanism exposes sensitive configuration data including database credentials and API keys in debug output",
        "Directory traversal vulnerability in {product} file upload component allows remote attacker to read arbitrary files on the server filesystem via path manipulation in filename parameter",
        "Server-side request forgery vulnerability in {product} webhook integration allows attacker to make internal network requests and potentially access internal services through crafted URL input",
        "Insecure deserialization vulnerability in {product} data processing module allows remote attacker to execute arbitrary code by submitting malicious serialized object in API request body",
    ]

products = ['Apache Struts', 'Nginx Proxy', 'WordPress Core', 'Spring Framework', 'Docker Engine',
            'Elasticsearch', 'Redis Server', 'MySQL Server', 'PostgreSQL', 'Node.js Express']
components = ['authentication', 'input parsing', 'session management', 'request routing',
              'file handling', 'database connector', 'template engine', 'logging subsystem']
versions = ['2.4.51', '3.1.0', '6.2.1', '5.3.27', '20.10.8', '7.17.0', '6.2.6', '8.0.28', '14.1', '4.18.2']

cve_descriptions = []
cve_ids = []
for i in range(80):
    template = np.random.choice(cve_templates)
    desc = template.format(
        product=np.random.choice(products),
        ver=np.random.choice(versions),
        component=np.random.choice(components)
    )
    cve_descriptions.append(desc)
    cve_ids.append(f'CVE-2024-{1000+i}')

# 3. Deskripsi TTP Threat Actor
ttp_templates = [
        "APT group conducts spear phishing campaign targeting defense sector employees using malicious Excel attachments with embedded macros that download additional payload from command and control server",
        "Cybercriminal organization deploys ransomware through compromised VPN credentials using legitimate remote access tools for lateral movement before encrypting critical business systems",
        "State-sponsored threat actor exploits zero-day vulnerability in perimeter firewall to establish persistent backdoor access enabling long-term intelligence collection from government networks",
        "Financially motivated actor conducts supply chain compromise by injecting malicious code into software update mechanism affecting thousands of downstream customers across multiple industries",
        "Hacktivist group performs distributed denial of service attacks against critical infrastructure targets using botnet of compromised IoT devices with volumetric traffic exceeding 100 Gbps",
        "Advanced persistent threat utilizes living off the land techniques leveraging PowerShell WMI and legitimate administrative tools to evade detection while conducting network reconnaissance",
        "Insider threat actor exfiltrates sensitive intellectual property through encrypted cloud storage uploads over extended period of six months bypassing data loss prevention controls",
        "Cyber espionage group targets healthcare organizations exploiting unpatched Exchange server vulnerabilities to deploy web shells and establish covert data extraction channels",
    ]

ttp_descriptions = []
ttp_ids = []
for i in range(60):
    ttp_descriptions.append(np.random.choice(ttp_templates))
    ttp_ids.append(f'TTP-2024-{500+i}')

# 4. Deskripsi SIEM Alert
alert_templates = [
        "Multiple failed login attempts detected for user administrator from IP {ip} total {n} attempts within {m} minutes possible brute force attack against Active Directory",
        "Anomalous data transfer volume detected from internal server {srv} to external destination {ip} exceeding baseline threshold by {x} times possible data exfiltration",
        "Malware signature match detected on endpoint {ep} file hash associated with known ransomware family immediate isolation recommended threat level critical",
        "Suspicious PowerShell execution detected on server {srv} encoded command line arguments consistent with living off the land binary and script techniques",
        "Unusual outbound connection from database server {srv} to previously unseen external IP {ip} on non-standard port {port} potential unauthorized data access",
        "Privilege escalation alert user account {user} granted domain admin rights outside of change management window investigate for potential account compromise",
        "Network beaconing activity detected from workstation {ep} regular interval connections to external IP {ip} every {m} minutes possible command and control communication",
        "Unauthorized application execution detected on endpoint {ep} unsigned binary attempting to modify system registry keys possible unauthorized software installation",
    ]

siem_alerts = []
alert_severities = []
for i in range(150):
    template = np.random.choice(alert_templates)
    alert = template.format(
        ip=f'10.{np.random.randint(0,256)}.{np.random.randint(0,256)}.{np.random.randint(1,255)}',
        srv=f'SRV-{np.random.choice(["WEB","DB","APP","MAIL","DNS"])}-{np.random.randint(1,20):02d}',
        ep=f'WS-{np.random.choice(["FIN","HR","ENG","EXEC"])}-{np.random.randint(1,50):03d}',
        user=f'user_{np.random.choice(["jsmith","admin","dbrown","klee","mwilson"])}',
        n=np.random.randint(10, 100),
        m=np.random.randint(1, 30),
        x=np.random.randint(3, 50),
        port=np.random.choice([4444, 8443, 9090, 1337, 31337])
    )
    siem_alerts.append(alert)
    alert_severities.append(np.random.choice(['Low', 'Medium', 'High', 'Critical'], p=[0.2, 0.35, 0.3, 0.15]))

# Buat DataFrame
df_firewall = pd.DataFrame({
    'log_id': [f'FW-{i:04d}' for i in range(len(firewall_logs))],
    'message': firewall_logs,
    'action': [m.split()[0] for m in firewall_logs]
})

df_cve = pd.DataFrame({
    'cve_id': cve_ids,
    'description': cve_descriptions,
    'product': [np.random.choice(products) for _ in range(len(cve_descriptions))]
})

df_ttp = pd.DataFrame({
    'ttp_id': ttp_ids,
    'description': ttp_descriptions
})

df_alerts = pd.DataFrame({
    'alert_id': [f'ALT-{i:04d}' for i in range(len(siem_alerts))],
    'description': siem_alerts,
    'severity': alert_severities
})

print("=== Dataset Summary ===")
print(f"Firewall Logs:  {len(df_firewall)} entries")
print(f"CVE Descriptions: {len(df_cve)} entries")
print(f"TTP Descriptions: {len(df_ttp)} entries")
print(f"SIEM Alerts:    {len(df_alerts)} entries")
print(f"Total text documents: {len(df_firewall) + len(df_cve) + len(df_ttp) + len(df_alerts)}")
print(f"\nSample firewall log:")
print(df_firewall['message'].iloc[0])
print(f"\nSample CVE description:")
print(df_cve['description'].iloc[0][:150] + '...')

---
## Soal 1: Preparing Text Data

Bersihkan dan tokenize firewall log messages:
1. Convert ke lowercase
2. Hapus IP addresses (regex)
3. Hapus punctuation
4. Tokenize menjadi individual words
5. Buat kolom `cleaned_tokens` berisi list of tokens

In [ ]:

def clean_log_text(text):
    text = text.lower()
    text = re.sub(r'\d+\.\d+\.\d+\.\d+', 'IP_ADDR', text)  # Replace IPs
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = re.sub(r'[/\\]', ' ', text)  # Remove path separators
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    return tokens

df_firewall['cleaned_tokens'] = df_firewall['message'].apply(clean_log_text)
df_firewall['cleaned_text'] = df_firewall['cleaned_tokens'].apply(' '.join)

print("=== Before Cleaning ===")
print(df_firewall['message'].iloc[0])
print(f"\n=== After Cleaning ===")
print(' '.join(df_firewall['cleaned_tokens'].iloc[0]))
print(f"\nToken count per log:")
print(df_firewall['cleaned_tokens'].apply(len).describe())
print(f"\nTotal unique tokens: {len(set().union(*df_firewall['cleaned_tokens'].values))}")

---
## Soal 2: Removing Stop Words

Hapus stop words dari CVE descriptions. Bandingkan jumlah kata sebelum dan sesudah. Kata apa yang paling sering muncul setelah stop words dihapus?

In [ ]:

stop_words = set(stopwords.words('english'))
# Tambahkan stop words spesifik security
security_stops = {'vulnerability', 'allows', 'attacker', 'remote', 'via', 'using', 'could', 'would'}
stop_words = stop_words.union(security_stops)

def remove_stopwords(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t not in stop_words and t not in string.punctuation]
    return tokens

df_cve['tokens_original'] = df_cve['description'].apply(lambda x: len(word_tokenize(x.lower())))
df_cve['tokens_filtered'] = df_cve['description'].apply(lambda x: len(remove_stopwords(x)))
df_cve['filtered_tokens'] = df_cve['description'].apply(remove_stopwords)

print("=== Stop Word Removal Stats ===")
print(f"Avg tokens before: {df_cve['tokens_original'].mean():.1f}")
print(f"Avg tokens after:  {df_cve['tokens_filtered'].mean():.1f}")
print(f"Reduction: {100 - df_cve['tokens_filtered'].mean()/df_cve['tokens_original'].mean()*100:.1f}%")

# Kata paling umum setelah filtering
all_filtered = [t for tokens in df_cve['filtered_tokens'] for t in tokens]
top_words = Counter(all_filtered).most_common(15)
print(f"\nTop 15 words after stopword removal:")
for word, count in top_words:
    print(f"  {word}: {count}")

---
## Soal 3: Analysing Part of Speech (POS)

In [ ]:

def analyze_pos(text):
    tokens = word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    return tagged

df_ttp['pos_tags'] = df_ttp['description'].apply(analyze_pos)

# Distribusi POS tags
all_tags = [tag for doc_tags in df_ttp['pos_tags'] for word, tag in doc_tags]
tag_dist = Counter(all_tags).most_common(10)
print("=== Top 10 POS Tags in TTP Descriptions ===")
pos_descriptions = {
    'NN': 'Noun (singular)', 'NNS': 'Noun (plural)', 'NNP': 'Proper noun',
    'VB': 'Verb (base)', 'VBD': 'Verb (past)', 'VBG': 'Verb (gerund)',
    'JJ': 'Adjective', 'RB': 'Adverb', 'DT': 'Determiner', 'IN': 'Preposition'
}
for tag, count in tag_dist:
    desc = pos_descriptions.get(tag, tag)
    print(f"  {tag:4s} ({desc:20s}): {count}")

# Ekstrak noun phrases
nouns = [word for doc_tags in df_ttp['pos_tags'] for word, tag in doc_tags if tag.startswith('NN')]
top_nouns = Counter(nouns).most_common(15)
print(f"\n=== Top 15 Nouns in TTP Descriptions ===")
for word, count in top_nouns:
    print(f"  {word}: {count}")

---
## Soal 4: Stemming & Lemmatisation

Terapkan stemming (PorterStemmer) dan lemmatisation (WordNetLemmatizer) pada SIEM alert descriptions. Bandingkan hasilnya. Mana yang lebih baik mempertahankan makna?

In [ ]:

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def get_pos_wordnet(tag):
    if tag.startswith('J'): return 'a'  # Adjective
    if tag.startswith('V'): return 'v'  # Verb
    if tag.startswith('N'): return 'n'  # Noun
    if tag.startswith('R'): return 'r'  # Adverb
    return 'n'

def stem_text(text):
    tokens = word_tokenize(text.lower())
    return [stemmer.stem(t) for t in tokens if t not in string.punctuation]

def lemmatize_text(text):
    tokens = word_tokenize(text.lower())
    tagged = nltk.pos_tag(tokens)
    return [lemmatizer.lemmatize(t, get_pos_wordnet(tag)) for t, tag in tagged if t not in string.punctuation]

# Test pada 3 alert pertama
for i in range(3):
    original = df_alerts['description'].iloc[i]
    stemmed = stem_text(original)
    lemmatized = lemmatize_text(original)
    print(f"\n=== Alert {i+1} ===")
    print(f"Original:   {original[:100]}...")
    print(f"Stemmed:    {' '.join(stemmed[:15])}...")
    print(f"Lemmatized: {' '.join(lemmatized[:15])}...")

# Terapkan ke semua alert
df_alerts['stemmed'] = df_alerts['description'].apply(stem_text)
df_alerts['lemmatized'] = df_alerts['description'].apply(lemmatize_text)

# Perbandingan: token unik
all_stemmed = set(t for doc in df_alerts['stemmed'] for t in doc)
all_lemmatized = set(t for doc in df_alerts['lemmatized'] for t in doc)
print(f"\n=== Vocabulary Size Comparison ===")
print(f"Original tokens:     {len(set(t for doc in df_alerts['description'].apply(lambda x: word_tokenize(x.lower())) for t in doc))}")
print(f"Stemmed unique:      {len(all_stemmed)}")
print(f"Lemmatized unique:   {len(all_lemmatized)}")
print(f"\nStemming mengurangi vocabulary lebih banyak tapi kehilangan makna (misal: 'detected' -> 'detect')")
print(f"Lemmatisation lebih preserve makna (misal: 'detected' -> 'detect', 'connections' -> 'connection')")

---
## Soal 5: Analysing Ngrams

Temukan bigram dan trigram paling umum dari semua CVE descriptions. Ini membantu mengidentifikasi pola vulnerability yang sering muncul.

In [ ]:

from nltk import ngrams

# Siapkan text: lowercase, tanpa stop words, tanpa punctuation
cve_clean = df_cve['description'].apply(lambda x: x.lower())
cve_tokens = cve_clean.apply(lambda x: [t for t in word_tokenize(x) if t not in stop_words and t not in string.punctuation])

# Bigram
all_bigrams = []
for tokens in cve_tokens:
    all_bigrams.extend(list(ngrams(tokens, 2)))
top_bigrams = Counter(all_bigrams).most_common(15)

print("=== Top 15 Bigrams in CVE Descriptions ===")
for (w1, w2), count in top_bigrams:
    print(f"  ({w1}, {w2}): {count}")

# Trigram
all_trigrams = []
for tokens in cve_tokens:
    all_trigrams.extend(list(ngrams(tokens, 3)))
top_trigrams = Counter(all_trigrams).most_common(10)

print(f"\n=== Top 10 Trigrams in CVE Descriptions ===")
for (w1, w2, w3), count in top_trigrams:
    print(f"  ({w1}, {w2}, {w3}): {count}")

# Analisis ngram untuk firewall logs
fw_tokens = df_firewall['cleaned_tokens']
fw_bigrams = []
for tokens in fw_tokens:
    fw_bigrams.extend(list(ngrams(tokens, 2)))
top_fw_bigrams = Counter(fw_bigrams).most_common(10)

print(f"\n=== Top 10 Bigrams in Firewall Logs ===")
for (w1, w2), count in top_fw_bigrams:
    print(f"  ({w1}, {w2}): {count}")

---
## Soal 6: Creating Word Clouds

Buat word cloud dari:
1. CVE descriptions (semua digabung)
2. TTP descriptions (semua digabung)

Stop words sudah difilter. Warna berbeda untuk masing-masing.

In [ ]:
# Install: pip install wordcloud

try:
    from wordcloud import WordCloud
    import matplotlib.pyplot as plt
    
    plt.style.use('seaborn-v0_8-darkgrid')
    
    # Word Cloud CVE
    cve_text = ' '.join(df_cve['filtered_tokens'].apply(' '.join))
    wc_cve = WordCloud(width=600, height=400, background_color='white',
                       max_words=100, colormap='Reds',
                       stopwords=stop_words,
                       collocations=False).generate(cve_text)
    
    # Word Cloud TTP
    ttp_tokens_clean = [[t.lower() for t in word_tokenize(doc) if t.lower() not in stop_words and t not in string.punctuation]
                        for doc in df_ttp['description']]
    ttp_text = ' '.join([' '.join(tokens) for tokens in ttp_tokens_clean])
    wc_ttp = WordCloud(width=600, height=400, background_color='white',
                       max_words=100, colormap='Blues',
                       stopwords=stop_words,
                       collocations=False).generate(ttp_text)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(wc_cve, interpolation='bilinear')
    axes[0].set_title('CVE Descriptions Word Cloud', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(wc_ttp, interpolation='bilinear')
    axes[1].set_title('TTP Descriptions Word Cloud', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig('/tmp/ch8_wordclouds.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Word Clouds saved to /tmp/ch8_wordclouds.png")
    
except ImportError:
    print("⚠️ wordcloud not installed. Run: pip install wordcloud")
    # Fallback: bar chart kata teratas
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    cve_words = [t for tokens in df_cve['filtered_tokens'] for t in tokens]
    top_cve = Counter(cve_words).most_common(12)
    axes[0].barh([w for w, c in top_cve[::-1]], [c for w, c in top_cve[::-1]], color='#e74c3c')
    axes[0].set_title('CVE: Top Words (Word Cloud alternative)', fontsize=12, fontweight='bold')
    
    ttp_words = [t.lower() for doc in df_ttp['description'] for t in word_tokenize(doc) if t.lower() not in stop_words and t not in string.punctuation]
    top_ttp = Counter(ttp_words).most_common(12)
    axes[1].barh([w for w, c in top_ttp[::-1]], [c for w, c in top_ttp[::-1]], color='#3498db')
    axes[1].set_title('TTP: Top Words (Word Cloud alternative)', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('/tmp/ch8_wordclouds_fallback.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Fallback bar charts saved to /tmp/ch8_wordclouds_fallback.png")

---
## Soal 7: Checking Term Frequency (TF-IDF)

Hitung TF-IDF untuk semua CVE descriptions. Temukan:
1. Top 10 terms dengan TF-IDF tertinggi (paling discriminative)
2. Bandingkan TF-IDF antara CVE tentang 'SQL injection' vs 'Remote code execution'

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Custom stop words
custom_stops = list(stop_words) + ['allows', 'attacker', 'allows']

vectorizer = TfidfVectorizer(
    max_features=200,
    stop_words=custom_stops,
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = vectorizer.fit_transform(df_cve['description'])
feature_names = vectorizer.get_feature_names_out()

# Average TF-IDF score per term
avg_tfidf = tfidf_matrix.mean(axis=0).A1
tfidf_df = pd.DataFrame({
    'term': feature_names,
    'avg_tfidf': avg_tfidf
}).sort_values('avg_tfidf', ascending=False)

print("=== Top 10 Terms by TF-IDF Score ===")
for _, row in tfidf_df.head(10).iterrows():
    print(f"  {row['term']:30s}: {row['avg_tfidf']:.4f}")

# Bandingkan CVE SQL injection vs RCE
sql_idx = df_cve[df_cve['description'].str.contains('SQL injection|sql injection', case=False)].index.tolist()
rce_idx = df_cve[df_cve['description'].str.contains('remote code execution', case=False)].index.tolist()

if sql_idx and rce_idx:
    print(f"\n=== TF-IDF: SQL Injection CVEs (n={len(sql_idx)}) ===")
    sql_tfidf = tfidf_matrix[sql_idx].mean(axis=0).A1
    sql_top = sorted(zip(feature_names, sql_tfidf), key=lambda x: -x[1])[:8]
    for term, score in sql_top:
        print(f"  {term:30s}: {score:.4f}")
    
    print(f"\n=== TF-IDF: RCE CVEs (n={len(rce_idx)}) ===")
    rce_tfidf = tfidf_matrix[rce_idx].mean(axis=0).A1
    rce_top = sorted(zip(feature_names, rce_tfidf), key=lambda x: -x[1])[:8]
    for term, score in rce_top:
        print(f"  {term:30s}: {score:.4f}")

# Shape matrix TF-IDF
print(f"\nTF-IDF Matrix Shape: {tfidf_matrix.shape} (documents x features)")
print(f"Sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.2%}")

---
## Soal 8: Checking Sentiments

Analisis sentiment dari SIEM alert descriptions. Apakah alert dengan severity 'Critical' memiliki sentiment lebih negatif? Bandingkan sentiment score per severity level.

In [ ]:
# Install: pip install textblob

try:
    from textblob import TextBlob
    
    def get_sentiment(text):
        blob = TextBlob(text)
        return blob.sentiment.polarity, blob.sentiment.subjectivity
    
    df_alerts[['polarity', 'subjectivity']] = pd.DataFrame(
        df_alerts['description'].apply(get_sentiment).tolist(),
        index=df_alerts.index
    )
    
    print("=== Sentiment Analysis by Severity ===")
    sentiment_by_sev = df_alerts.groupby('severity').agg(
        avg_polarity=('polarity', 'mean'),
        avg_subjectivity=('subjectivity', 'mean'),
        count=('alert_id', 'count')
    )
    print(sentiment_by_sev.round(4))
    
    # Visualisasi
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    colors = {'Low': '#27ae60', 'Medium': '#f39c12', 'High': '#e67e22', 'Critical': '#e74c3c'}
    sev_colors = [colors[s] for s in sentiment_by_sev.index]
    
    axes[0].bar(sentiment_by_sev.index, sentiment_by_sev['avg_polarity'], color=sev_colors, alpha=0.8)
    axes[0].axhline(0, color='black', linewidth=0.5)
    axes[0].set_title('Avg Polarity by Severity', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Polarity (-1 to +1)')
    
    axes[1].bar(sentiment_by_sev.index, sentiment_by_sev['avg_subjectivity'], color=sev_colors, alpha=0.8)
    axes[1].set_title('Avg Subjectivity by Severity', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Subjectivity (0 to 1)')
    
    plt.tight_layout()
    plt.savefig('/tmp/ch8_sentiment.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✅ Sentiment chart saved to /tmp/ch8_sentiment.png")
    
except ImportError:
    print("⚠️ textblob not installed. Run: pip install textblob")
    print("Skipping sentiment analysis.")

---
## Soal 9: Performing Topic Modelling (LDA)

Gunakan **LDA (Latent Dirichlet Allocation)** untuk menemukan topik tersembunyi dalam CVE descriptions. Set 4 topics. Interpretasikan setiap topik berdasarkan kata-kata teratas.

In [ ]:
# Install: pip install gensim

try:
    import gensim
    import gensim.corpora as corpora
    from gensim.models import LdaModel
    
    # Siapkan corpus
    texts = df_cve['filtered_tokens'].tolist()
    texts = [[t for t in doc if len(t) > 2] for doc in texts]  # Remove short tokens
    
    # Buat dictionary dan corpus
    dictionary = corpora.Dictionary(texts)
    dictionary.filter_extremes(no_below=2, no_above=0.8)
    corpus = [dictionary.doc2bow(text) for text in texts]
    
    print(f"Dictionary size: {len(dictionary)}")
    print(f"Corpus size: {len(corpus)} documents")
    
    # Latih LDA
    num_topics = 4
    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        random_state=42,
        passes=10,
        per_word_topics=True
    )
    
    print(f"\n=== LDA Topics (num_topics={num_topics}) ===")
    topic_labels = []
    for idx, topic in lda_model.print_topics(num_words=8):
        print(f"\nTopic {idx}:")
        words = []
        for word, weight in lda_model.show_topic(idx, topn=8):
            print(f"  {word:20s}: {weight:.4f}")
            words.append(word)
        topic_labels.append(' '.join(words[:4]))
    
    # Assign topik dominan ke setiap dokumen
    doc_topics = []
    for bow in corpus:
        topic_dist = lda_model.get_document_topics(bow)
        dominant = max(topic_dist, key=lambda x: x[1])
        doc_topics.append(dominant[0])
    
    df_cve['dominant_topic'] = doc_topics
    df_cve['topic_label'] = df_cve['dominant_topic'].map(
        {i: f'Topic {i}: ' + topic_labels[i][:50] for i in range(num_topics)}
    )
    
    print(f"\n=== Document Distribution by Topic ===")
    print(df_cve['dominant_topic'].value_counts().sort_index())
    
    # Visualisasi
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    topic_counts = df_cve['dominant_topic'].value_counts().sort_index()
    ax.bar(topic_counts.index.astype(str), topic_counts.values, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'], alpha=0.8)
    ax.set_title('LDA Topic Distribution in CVEs', fontsize=12, fontweight='bold')
    ax.set_xlabel('Topic')
    ax.set_ylabel('Number of CVEs')
    plt.tight_layout()
    plt.savefig('/tmp/ch8_lda_topics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✅ LDA chart saved to /tmp/ch8_lda_topics.png")
    
except ImportError:
    print("⚠️ gensim not installed. Run: pip install gensim")
    print("Skipping topic modelling.")

---
## Soal 10: Choosing Optimal Number of Topics

Cari jumlah topic optimal dengan menghitung **Coherence Score** untuk num_topics = 2 hingga 8. Pilih jumlah topic dengan coherence tertinggi.

In [ ]:
# Install: pip install gensim

try:
    from gensim.models import CoherenceModel
    from gensim.models import LdaModel
    import matplotlib.pyplot as plt
    
    # Siapkan (pakai ulang dari Soal 9)
    texts = df_cve['filtered_tokens'].tolist()
    texts = [[t for t in doc if len(t) > 2] for doc in texts]
    dictionary = corpora.Dictionary(texts)
    dictionary.filter_extremes(no_below=2, no_above=0.8)
    corpus = [dictionary.doc2bow(text) for text in texts]
    
    # Test berbagai jumlah topik
    topic_range = range(2, 9)
    coherence_scores = []
    perplexity_scores = []
    
    print("=== Testing Coherence for Different Topic Counts ===")
    for num_topics in topic_range:
        lda = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=num_topics,
            random_state=42,
            passes=10
        )
        
        coherence = CoherenceModel(
            model=lda,
            texts=texts,
            dictionary=dictionary,
            coherence='c_v'
        ).get_coherence()
        
        perplexity = lda.log_perplexity(corpus)
        coherence_scores.append(coherence)
        perplexity_scores.append(perplexity)
        
        print(f"  Topics: {num_topics} | Coherence: {coherence:.4f} | Perplexity: {perplexity:.2f}")
    
    # Cari yang optimal
    best_idx = coherence_scores.index(max(coherence_scores))
    best_topics = list(topic_range)[best_idx]
    
    print(f"\n📌 Optimal number of topics: {best_topics} (Coherence: {coherence_scores[best_idx]:.4f})")
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(topic_range, coherence_scores, 'o-', color='#e74c3c', linewidth=2, markersize=8)
    axes[0].axvline(best_topics, color='green', linestyle='--', label=f'Optimal: {best_topics}')
    axes[0].set_title('Coherence Score vs Number of Topics', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Number of Topics')
    axes[0].set_ylabel('Coherence Score (C_v)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(topic_range, perplexity_scores, 'o-', color='#3498db', linewidth=2, markersize=8)
    axes[1].set_title('Perplexity vs Number of Topics', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Number of Topics')
    axes[1].set_ylabel('Perplexity (lower is better)')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/tmp/ch8_coherence.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✅ Coherence chart saved to /tmp/ch8_coherence.png")
    
except ImportError:
    print("⚠️ gensim not installed. Run: pip install gensim")
    print("Skipping optimal topic selection.")